# Jupyter Notebook: SLC9000 API #

## Import modules and define connection settings

This cell sets up the environment for talking to the SLC9000 device. It imports required Python libraries, reads credentials from environment variables, configures the `requests` session (TLS verification, headers, timeout), and defines the base URL and other constants used by the REST API calls.


In [47]:
import json
import os
import re
import requests
import time
from pprint import pprint
from tabulate import tabulate

TIMEOUT = 10
VERIFY = False

# Create & use these environment variables or the load_config function
# USERNAME = os.environ.get("SLC9K_USER", "sysadmin")
# PASSWORD = os.environ.get("SLC9K_PW", "")
#
USERNAME = ""
PASSWORD = ""
PERCEPXION_HOST = ""

requests.packages.urllib3.disable_warnings()
session = requests.Session()
session.verify = VERIFY
session.timeout = TIMEOUT
session.headers.update(
    {
        "Content-Type": "application/json",
        "Accept": "application/json",
   }
)
session.base_url = "https://10.40.21.41"

### Load configuration from JSON file

In [48]:
def load_config(filename="config.json"):
    """
    Load connection and authentication settings from a JSON config file.

    This function reads the given JSON configuration file and updates the
    global TIMEOUT, VERIFY, USERNAME, PASSWORD, and PERCEPXION_URL variables. It allows the
    SLC9000 API demo to be configured without hard‑coding values in the
    notebook or relying solely on environment variables.

    The expected JSON structure is:
        {
            "TIMEOUT": 10,
            "VERIFY": false,
            "USERNAME": "sysadmin",
            "PASSWORD": "ciscolive",
            "PERCEPXION_URL": "api.percepxion.ai"
        }

    Args:
        filename: Path to the JSON configuration file to load. Defaults to
            "config.json" in the current working directory.
    """
    global TIMEOUT, VERIFY, USERNAME, PASSWORD, PERCEPXION_HOST
    with open(filename, "r") as f:
        config = json.load(f)

    TIMEOUT = config.get("TIMEOUT", TIMEOUT)
    VERIFY = config.get("VERIFY", VERIFY)
    USERNAME = config.get("USERNAME", USERNAME)
    PASSWORD = config.get("PASSWORD", PASSWORD)
    PERCEPXION_HOST = config.get("PERCEPXION_URL", PERCEPXION_HOST)

## Helper function for SLC9000 REST API calls

In [49]:
def api_call(
    method: str,
    path: str,
    payload: dict | None = None
):
    """
    Perform an HTTP request to the SLC9000 API using the shared session.

    This helper builds a full URL from the session's base_url and the given
    path, sends the request with the specified HTTP method, and handles basic
    error reporting. On success it returns the underlying requests.Response
    object; on failure it logs the error and returns None.

    Args:
        method: HTTP method to use, e.g. "GET", "POST", "PUT", or "DELETE".
        path: API path to append to session.base_url, e.g. "/api/v2/system/status".
        payload: Optional JSON-serializable dictionary to send as the request body
            for POST, PUT, and DELETE requests.

    Returns:
        A requests.Response object if the request succeeds; otherwise None.

    Raises:
        ValueError: If an unsupported HTTP method is provided.
    """
    url = f"{session.base_url}{path}"
    try:
        if method == "GET":
            response = session.get(url, timeout=TIMEOUT)
        elif method == "POST":
            response = session.post(url, json=payload or {}, timeout=TIMEOUT)
        elif method == "PATCH":
            response = session.patch(url, json=payload or {}, timeout=TIMEOUT)
        elif method == "PUT":
            response = session.put(url, json=payload or {}, timeout=TIMEOUT)
        elif method == "DELETE":
            response = session.delete(url, json=payload or {}, timeout=TIMEOUT)
        else:
            raise ValueError(f"Unsupported method: {method}")

        response.raise_for_status()
        print(f"{url} successful ({response.status_code})")
        return response
    except requests.exceptions.HTTPError as http_err:
        body = response.text if "response" in locals() else "<no response>"
        print(f"HTTP error: {http_err} - Response: {body}")
    except requests.exceptions.RequestException as err:
        print(f"Request error: {err}")
    return None

# All API endpoints as of 9.7.0.0R17 #

In [50]:
# ------------------------
# User Management
# ------------------------

def user_login(username: str, password: str):
    """
    POST /user/login
    User login for all roles (supports 2FA challenge flow).
    """
    response = api_call(
        "POST",
        "/api/v2/user/login",
        {"username": username, "password": password},
    )
    if response is None:
        return None
        
    data = response.json()
    token = data.get("token")
    if token:
        session.headers.update({"X-auth-token": token})
    return response


def user_logout():
    """
    DELETE /user/login
    Logout from an API session.
    """
    return api_call(
        "DELETE",
        "/api/v2/user/login",
    )


def sessions():
    """
    GET /sessions
    Get list of active web, API and Web Terminal sessions.
    """
    return api_call(
        "GET",
        "/api/v2/sessions",
    )


def session_info(session_id: str):
    """
    GET /sessions/{SESSIONID}
    Retrieve info about an active session.
    """
    return api_call(
        "GET",
        f"/api/v2/sessions/{session_id}",
    )


def session_terminate(session_id: str):
    """
    DELETE /sessions/{SESSIONID}
    Terminate an active web, Percepxion web, API or Web Terminal session.
    """
    return api_call(
        "DELETE",
        f"/api/v2/sessions/{session_id}",
    )


def sysadmin_get():
    """
    GET /users/sysadmin
    Get sysadmin user attributes.
    """
    return api_call(
        "GET",
        "/api/v2/users/sysadmin",
    )


def sysadmin_update_password(new_password: str):
    """
    PATCH /users/sysadmin
    Update sysadmin user password.
    """
    payload = {"new_password": new_password}
    return api_call(
        "PATCH",
        "/api/v2/users/sysadmin",
        payload,
    )


# ------------------------
# Network
# ------------------------

def network_interfaces():
    """
    GET /network/interfaces
    Get Eth1 IP addresses, gateway, DNS, NTP, etc.
    """
    return api_call(
        "GET",
        "/api/v2/network/interfaces",
    )


def network_interfaces_set(
    eth1_ipv4: str,
    eth1_mask: str,
    eth1_ipv6: str,
    gateway_ipv4: str,
    gateway_ipv6: str,
    dns_1: str,
    ntp_server_1: str,
    **optional_fields,
):
    """
    PUT /network/interfaces
    Set Eth1 IP addresses, gateway, DNS server, NTP server.

    Required fields come from NetworkInterfaces schema; extra optional
    fields (eth2_*, eth3_*, eth4_*) can be passed via **optional_fields.
    """
    payload = {
        "eth1_ipv4": eth1_ipv4,
        "eth1_mask": eth1_mask,
        "eth1_ipv6": eth1_ipv6,
        "gateway_ipv4": gateway_ipv4,
        "gateway_ipv6": gateway_ipv6,
        "dns_1": dns_1,
        "ntp_server_1": ntp_server_1,
    }
    payload.update(optional_fields)
    return api_call(
        "PUT",
        "/api/v2/network/interfaces",
        payload,
    )


# ------------------------
# System
# ------------------------

def system_identity():
    """
    GET /system/identity
    Retrieve basic system information (hostname, device ID, S/N, site, rack/row).
    """
    return api_call(
        "GET",
        "/api/v2/system/identity",
    )


def system_identity_set(
    hostname: str,
    site_tag: str,
    rack_row: int,
    rack_cluster: int,
    rack: int,
):
    """
    POST /system/identity
    Set basic system information - hostname, site, rack/row.
    """
    payload = {
        "hostname": hostname,
        "site_tag": site_tag,
        "rack_row": rack_row,
        "rack_cluster": rack_cluster,
        "rack": rack,
    }
    return api_call(
        "POST",
        "/api/v2/system/identity",
        payload,
    )


def system_status():
    """
    GET /system/status
    Get top level system status and health for all SLC components.
    """
    return api_call(
        "GET",
        "/api/v2/system/status",
    )


def system_version():
    """
    GET /system/version
    Get hardware, software and model information.
    """
    return api_call(
        "GET",
        "/api/v2/system/version",
    )


def system_ztp():
    """
    GET /system/ztp
    Get results of device onboarding (ZTP/bootstrap/Percepxion).
    """
    return api_call(
        "GET",
        "/api/v2/system/ztp",
    )


def system_reboot():
    """
    POST /system/reboot
    Reboot the device.
    """
    return api_call(
        "POST",
        "/api/v2/system/reboot",
    )


# ------------------------
# Firmware
# ------------------------

def firmware_version():
    """
    GET /firmware/version
    Get firmware version on both banks, current boot bank, and last update status.
    """
    return api_call(
        "GET",
        "/api/v2/firmware/version",
    )


def firmware_bootbank():
    """
    GET /firmware/bootbank
    Get the active SLC boot bank (1 or 2).
    """
    return api_call(
        "GET",
        "/api/v2/firmware/bootbank",
    )


def firmware_bootbank_set(bank: int):
    """
    PUT /firmware/bootbank
    Set active boot bank (after next reboot).
    """
    payload = {"bank": bank}
    return api_call(
        "PUT",
        "/api/v2/firmware/bootbank",
        payload,
    )


def firmware_check():
    """
    GET /firmware/check
    Check for available firmware updates and compare to installed versions.
    """
    return api_call(
        "GET",
        "/api/v2/firmware/check",
    )


def firmware_update_multipart(
    data: dict,
    file_path: str | None = None,
):
    """
    POST /firmware/update
    Update firmware on the alternate boot bank.

    This uses multipart/form-data, so we bypass api_call and use the shared
    `session` directly.

    - If data includes "file_url", the remote URL is used and file_path may be None.
    - If "file_url" is not provided, file_path must point to the local firmware file.
    """
    url = f"{session.base_url}/api/v2/firmware/update"

    # data field is JSON string
    multipart_data = {"data": json.dumps(data)}
    pprint(multipart_data)
    print(multipart_data["data"].get("file_url"))

    files = None
    if "file_url" not in data:
        if not file_path:
            raise ValueError("file_path is required when data does not contain 'file_url'")
        files = {"file": open(file_path, "rb")}

    try:
        resp = session.post(
            url,
            data=multipart_data,
            files=files,
            timeout=TIMEOUT,
            verify=session.verify,
        )
        resp.raise_for_status()
        print(f"{url} successful ({resp.status_code})")
        return resp
    except requests.exceptions.HTTPError as http_err:
        body = resp.text if "resp" in locals() else "<no response>"
        print(f"HTTP error: {http_err} - Response: {body}")
    except requests.exceptions.RequestException as err:
        print(f"Request error: {err}")
    finally:
        if files and "file" in files:
            files["file"].close()
    return None


def firmware_update_status():
    """
    GET /firmware/update_status
    Get status of the current in‑progress firmware update.
    """
    return api_call(
        "GET",
        "/api/v2/firmware/update_status",
    )


def firmware_log():
    """
    GET /firmware/log
    Get log from latest completed firmware update.
    """
    return api_call(
        "GET",
        "/api/v2/firmware/log",
    )


# ------------------------
# Config
# ------------------------

def config_batch(commands: str):
    """
    POST /config/batch
    Import configuration as CLI commands.
    """
    payload = {"commands": commands}
    return api_call(
        "POST",
        "/api/v2/config/batch",
        payload,
    )

    
def config_baseline_restore():
    """
    POST /config/baseline
    Restore configuration acquired via ZTP/bootstrap/Percepxion.
    """
    return api_call(
        "POST",
        "/api/v2/config/baseline",
    )


def config_commands():
    """
    GET /config/commands
    Get SLC configuration as a series of CLI commands.
    """
    return api_call(
        "GET",
        "/api/v2/config/commands",
    )


def config_compare():
    """
    GET /config/compare
    Compare running configuration to a saved baseline config.
    (Note: endpoint is currently not implemented per spec.)
    """
    return api_call(
        "GET",
        "/api/v2/config/compare",
    )


def config_factory_reset():
    """
    POST /config/factory_reset
    Perform a factory reset followed by an immediate reboot.
    """
    return api_call(
        "POST",
        "/api/v2/config/factory_reset",
    )


def config_edit(selected_config_groups: list[str] | None = None):
    """
    POST /config/edit
    Returns device configuration for subsequent editing.

    selected_config_groups: list of group names (e.g. ["servssh"]) or
    ["ALLGROUPS"] to retrieve all.
    """
    payload = {}
    if selected_config_groups is not None:
        payload = {
            "selected_config_groups": [
                {"name": name} for name in selected_config_groups
            ]
        }
    return api_call(
        "POST",
        "/api/v2/config/edit",
        payload,
    )


def config_save(config_record: list[dict]):
    """
    POST /config/save
    Apply configuration changes to the device.

    config_record: list of objects of the form:
      {
        "name": "servssh",
        "field": [
          {"name": "servssh_enablelogins", "value": "Enable"},
          ...
        ]
      }
    """
    payload = {"config_record": config_record}
    return api_call(
        "POST",
        "/api/v2/config/save",
        payload,
    )


# ------------------------
# Ports
# ------------------------

def ports(status_filter: str | None = None):
    """
    GET /ports
    Get all serial ports status, optionally filtered by status substring.
    """
    path = "/api/v2/ports"
    if status_filter:
        path += f"?status_filter={status_filter}"
    return api_call(
        "GET",
        path,
    )


def port_status(port_id: str | int):
    """
    GET /ports/{PORTID}/status
    Retrieve state of serial port by port number or port name.
    """
    return api_call(
        "GET",
        f"/api/v2/ports/{port_id}/status",
    )


def port_action(port_id: str | int, payload: dict):
    """
    POST /ports/{ID}/action
    Send command to port. (Not implemented on device per spec.)
    """
    return api_call(
        "POST",
        f"/api/v2/ports/{port_id}/action",
        payload,
    )


# ------------------------
# Connections
# ------------------------

def connections():
    """
    GET /connections
    Get a list of active IP and direct connections to serial ports.
    """
    return api_call(
        "GET",
        "/api/v2/connections",
    )


# ------------------------
# Managed Devices
# ------------------------

def managed_devices(connection_filter: str | None = None):
    """
    GET /managed_devices
    Get all managed devices, optionally filtered by connection type:
    'serial' or 'ethernet'.
    """
    path = "/api/v2/managed_devices"
    if connection_filter:
        path += f"?connection_filter={connection_filter}"
    return api_call(
        "GET",
        path,
    )


def managed_device_status(device_id: str, device_type: str | None = None):
    """
    GET /managed_devices/{ID}/status
    Retrieve info about a single managed device.

    device_id: managed device name, serial port number, or switch port number.
    device_type: optional 'serial' or 'ethernet' to filter by connection type.
    """
    path = f"/api/v2/managed_devices/{device_id}/status"
    if device_type:
        path += f"?type={device_type}"
    return api_call(
        "GET",
        path,
    )


# ------------------------
# Cellular
# ------------------------

def cellular_status():
    """
    GET /cellular/status
    Detailed cellular modem status (if present).
    """
    return api_call(
        "GET",
        "/api/v2/cellular/status",
    )


### Initialize global variables

In [51]:
load_config()

print(f"TIMEOUT={TIMEOUT}")
print(f"VERIFY={VERIFY}")

TIMEOUT=10
VERIFY=False


## Authenticate and obtain API session token

This section logs in to the SLC9000 using the provided username and password. On success, the device returns a JSON response containing a session `token`, its `expires_in` lifetime (in seconds), and user details. The token is used by the REST API to identify and authorize subsequent requests made during this session.

In [52]:
login = user_login(username=USERNAME, password=PASSWORD)

if login is not None:
    pprint(login.json())

https://10.40.21.41/api/v2/user/login successful (200)
{'authenticated': 'Local Users',
 'expires_in': 1800,
 'token': 'e7c720a2-54ef-4a9c-9028-f6394e5ae945',
 'user': {'allow_dialback': False,
          'break_seq': '\\x1bB',
          'clear_ports': '1-32,U1,U2',
          'data_ports': '1-32,U1,U2',
          'dialback_number': 'null',
          'escape_seq': '\\x1bA',
          'group': 'Administrators',
          'listen_ports': '1-32,U1,U2',
          'permissions': 'ad,nt,sv,dt,lu,ra,um,dp,ub,rs,fc,dr,sn,wb,sk,po,do,md,rp,sw',
          'power_outlets': '1-8',
          'uid': 0,
          'username': 'sysadmin'}}


### List active API sessions

In [53]:
get_active_sessions = sessions()

if get_active_sessions is not None:
    data = get_active_sessions.json()
    results = data.get("results", [])
    sorted_results = sorted(results, key=lambda s: s["login_time"])

    rows = [
        [
            s.get("id", ""),
            s.get("session_type", ""),
            s.get("username", ""),
            s.get("remote_ip", ""),
            s.get("login_time", ""),
        ]
        for s in sorted_results
    ]

    headers = ["ID", "Type", "Username", "Remote IP", "Login Time"]
    print(tabulate(rows, headers=headers, tablefmt="github"))


https://10.40.21.41/api/v2/sessions successful (200)
|   ID | Type     | Username   | Remote IP    | Login Time           |
|------|----------|------------|--------------|----------------------|
| 4865 | REST API | sysadmin   | 10.40.21.223 | 2026-06-26T11:22:19Z |


### Check current firmware boot bank

In [54]:
get_firmware_bootbank = firmware_bootbank()

if get_firmware_bootbank is not None:
    pprint(get_firmware_bootbank.json())

https://10.40.21.41/api/v2/firmware/bootbank successful (200)
{'bank': 1}


### Inspect firmware update log

In [55]:
get_firmware_log = firmware_log()

if get_firmware_log is not None:
    pprint(get_firmware_log.json())

https://10.40.21.41/api/v2/firmware/log successful (200)
{'logs': 'Update Start: 06/20/26 14:28\n'
         'Current Bank Firmware Version: 9.7.0.0R18\n'
         'Alternate Bank Firmware Version: 9.7.0.0R17\n'
         'Initiating firmware update via Percepxion with '
         'slc9update-9.7.0.0R18.tgz.\n'
         'Saving current configuration to before_062026_1428-slc9cfg.tgz.\n'
         'Running slc9update-9.7.0.0R18.tgz ROM firmware update.\n'
         'Checking firmware version dependencies...\n'
         'Current running firmware version is 9.7.0.0, update version is '
         '9.7.0.0...\n'
         'Current running firmware version 9.7.0.0 meets minimum firmware '
         'requirements.\n'
         'Firmware upgrade started\n'
         "bootbank_kcmdline: 'Dual Bank 2'  bootbank_environment: 'Dual Bank "
         "2'\n"
         'image 1 type: gzipped ext4 rootfs image\n'
         'image 1 version: 9.7.0.0R18  (current: 9.7.0.0R17)\n'
         'image 1 product code: SE  (c

### Check for available firmware updates

In [56]:
get_firmware_check = firmware_check()

if get_firmware_check is not None:
    pprint(get_firmware_check.json())

https://10.40.21.41/api/v2/firmware/check successful (200)
{'latest_version': '9.8.0.0R7',
 'latest_version_notes': 'https://update.lantronix.com/SLC9000/released/SLC9000_v9.8.0.0_relnotes.txt',
 'latest_version_url': 'https://update.lantronix.com/SLC9000/released/slc9update-9.8.0.0R7.tgz',
 'up_to_date': False}


### Retrieve system software versions

In [67]:
get_system_version = system_version()

if get_system_version is not None:
    pprint(get_system_version.json())

https://10.40.21.41/api/v2/system/version successful (200)
{'bootloader_version': '2.0.0.0R12',
 'current_firmware_version': '9.7.0.0R18',
 'ec_version': '2.1',
 'io_module_revisions': '16SPF, 16UBB, 16ESB',
 'io_module_types': 'RJ45-16, USB-16, ETH-16',
 'main_board_version': 'unknown',
 'model': 'SLC9032',
 'power_supplies': 'AC, 2 power supplies',
 'sw_dnsmasq_version': '2.90',
 'sw_expect_version': '5.45.4',
 'sw_kernel_version': '6.6.52',
 'sw_ldap_version': '153',
 'sw_ntp_version': '4.2.8p18@1.4062-o',
 'sw_python_version': '3.13.2',
 'sw_radius_version': '3.0.0',
 'sw_rip_version': '9.1.3',
 'sw_ssh_version': 'OpenSSH_10.0p2, OpenSSL 3.4.1 11 Feb 2025',
 'sw_syslog_version': '2.7.1',
 'sw_tacacs_version': '1.6.0',
 'sw_tcl_version': '8.6',
 'sw_telnet_version': 'netkit-telnet-0.17',
 'sw_tls_version': 'OpenSSL 3.4.1 11 Feb 2025 (Library: OpenSSL 3.4.1 11 Feb '
                   '2025)',
 'sw_ttyd_version': '1.7.7',
 'sw_vpn_version': 'strongSwan U5.9.14/K6.6.52',
 'sw_webserve

### Check Zero Touch Provisioning (ZTP) status

In [68]:
get_system_ztp = system_ztp()

# if get_system_ztp is not None:
#     pprint(get_system_ztp.json())

if get_system_ztp is not None:
    data = get_system_ztp.json()

    print(f"ZTP Status:      {data.get('status')}")
    print(f"DHCP Status:     {data.get('dhcp_status')}")
    print(f"Config Status:   {data.get('config_status')}")
    print(f"Firmware Status: {data.get('firmware_status')}")
    print(f"Last Operation:  {data.get('last_operation')}")

    # Show only Percepxion-relevant log entries
    px_lines = [
        l for l in data.get('log', [])
        if l.startswith('PX:') or l.startswith('ZTP &')
    ]
    if px_lines:
        print("\nPercepxion events:")
        for line in px_lines:
            print(f"  {line}")

https://10.40.21.41/api/v2/system/ztp successful (200)
ZTP Status:      Not_Run
DHCP Status:     Success
Config Status:   Not_Attempted
Firmware Status: Not_Required
Last Operation:  2026-06-08T13:07:33.200000Z

Percepxion events:
  ZTP & Bootstrap Onboarding Start: 06/06/26 20:04
  ZTP & Bootstrap Onboarding Start: 06/06/26 20:13
  PX: Registered to cloud percepxion.ai at 06/06/26 20:14
  PX: Checking for configuration update at 06/06/26 20:14
  PX: Checking for firmware update at 06/06/26 20:14
  PX: Registered to cloud gopercepxion.ai at 06/08/26 13:07
  PX: Checking for configuration update at 06/08/26 13:07
  PX: Checking for firmware update at 06/08/26 13:07


### Detect Percepxion cloud endpoint

In [59]:
# Auto-detect from ZTP log; config.json PERCEPXION_URL takes precedence if set
if not PERCEPXION_HOST and get_system_ztp is not None:
    for line in get_system_ztp.json().get('log', []):
        m = re.search(r'Registered to cloud (\S+) at', line)
        if m:
            PERCEPXION_HOST = m.group(1)
            break

if PERCEPXION_HOST:
    print(f'Percepxion cloud:    {PERCEPXION_HOST}')
    print(f'Device dashboard:    https://{PERCEPXION_HOST}')
else:
    print('Percepxion host not detected, set PERCEPXION_URL in config.json if needed')

Percepxion cloud:    api.gopercepxion.ai
Device dashboard:    https://api.gopercepxion.ai


### Monitor system health and hardware status

In [60]:
get_system_status = system_status()

if get_system_status is not None:
    pprint(get_system_status.json())

https://10.40.21.41/api/v2/system/status successful (200)
{'cell_link': 'null',
 'console_port': 'Connected',
 'eth1_link': 'Up',
 'eth2_link': 'Up',
 'eth3_link': 'Down',
 'eth4_link': 'Down',
 'ps1': 'Failed',
 'ps2': 'Ok',
 'temperature': 54,
 'uptime': 81982,
 'warranty_end_date': 'Jan 1, 2000'}


### Show device identity and metadata

In [61]:
get_system_identity = system_identity()

if get_system_identity is not None:
    pprint(get_system_identity.json())

https://10.40.21.41/api/v2/system/identity successful (200)
{'device_id': '00204ADCOZG6CHHBENW6GWLKXKA58G05',
 'hostname': 'slc9000-dg-02',
 'rack': '1',
 'rack_cluster': '1',
 'rack_row': '1',
 'serial_number': '000F2C030AE0',
 'site_tag': ''}


### Display network interface configuration

In [62]:
get_network_interfaces = network_interfaces()

if get_network_interfaces is not None:
    pprint(get_network_interfaces.json())

https://10.40.21.41/api/v2/network/interfaces successful (200)
{'dns_1': '10.40.21.1',
 'eth1_ipv4': '10.40.21.41',
 'eth1_ipv6': 'fdbc:284b:d3c0:72d5:020f:2cff:fe03:0ae0/64',
 'eth1_link': 'Up',
 'eth1_mask': '255.255.255.0',
 'eth2_ipv4': '192.168.1.102',
 'eth2_ipv6': 'fda3:2473:ff62:0000:0200:00ff:fe00:0002/64',
 'eth2_link': 'Up',
 'eth2_mask': '255.255.255.0',
 'eth3_ipv4': '',
 'eth3_ipv6': '',
 'eth3_link': 'Down',
 'eth3_mask': '',
 'eth4_ipv4': '',
 'eth4_ipv6': '',
 'eth4_link': 'Down',
 'eth4_mask': '',
 'gateway_ipv4': '10.40.21.1',
 'gateway_ipv6': '',
 'ntp_server_1': None}


### Inventory all 32 serial ports

Every serial port is addressable via the API, physical type (RJ45 or USB), current status, and any attached managed device. The `status_filter` query parameter narrows the response to ports whose status contains a given string, e.g. `"Managed"` or `"SSH"`.

In [69]:
get_ports = ports()

if get_ports is not None:
    print(f"\nTotal Ports: {get_ports.json()['total_ports']}\n")
    rows = [
        [
            p.get("id", ""),
            p.get("name", ""),
            p.get("type",""),
            p.get("status", ""),
        ]
        for p in get_ports.json()['ports']
    ]
    headers = ["ID", "Name", "Type", "Status"]
    print(tabulate(rows, headers=headers, tablefmt="github"))

https://10.40.21.41/api/v2/ports successful (200)

Total Ports: 32

|   ID | Name       | Type   | Status   |
|------|------------|--------|----------|
|    1 | MikroTik   | rj45   | Idle     |
|    2 | C892FSP-K9 | rj45   | Idle     |
|    3 | Port-03    | rj45   | Idle     |
|    4 | Port-04    | rj45   | Idle     |
|    5 | Port-05    | rj45   | Idle     |
|    6 | Port-06    | rj45   | Idle     |
|    7 | Port-07    | rj45   | Idle     |
|    8 | Port-08    | rj45   | Idle     |
|    9 | Port-09    | rj45   | Idle     |
|   10 | Port-10    | rj45   | Idle     |
|   11 | Port-11    | rj45   | Idle     |
|   12 | Port-12    | rj45   | Idle     |
|   13 | Port-13    | rj45   | Idle     |
|   14 | Port-14    | rj45   | Idle     |
|   15 | Port-15    | rj45   | Idle     |
|   16 | Port-16    | rj45   | Idle     |
|   17 | Port-17    | usb    | Idle     |
|   18 | Port-18    | usb    | Idle     |
|   19 | Port-19    | usb    | Idle     |
|   20 | Port-20    | usb    | Idle     |
|   21 |

### Active connections to serial ports

Lists every active IP or direct connection currently open to a device port, who is connected, from where, for how long, and whether they authenticated. In a real OOB incident, this shows which engineers are actively working the problem.

In [73]:
get_connections = connections()

if get_connections is not None:
    total_connected = len(get_connections.json()['list'])
    print(f"\nTotal connections: {total_connected}...\n")
    rows = [
        [
            c.get("id", ""),
            c.get("description", ""),
            c.get("source_ip", "n/a"),
            c.get("username", "n/a"),
            c.get("duration") // 60
        ]
        for c in get_connections.json()['list']
    ]
    headers = ["ID", "Description", "Source IP", "Username", "Duration"]
    print(tabulate(rows, headers=headers, tablefmt="github"))

https://10.40.21.41/api/v2/connections successful (200)

Total connections: 1...

|   ID | Description                  | Source IP   | Username   |   Duration |
|------|------------------------------|-------------|------------|------------|
|    2 | Console Port to Command Line |             |            |       1368 |


### Managed devices discovered on serial ports

The SLC9000 fingerprints and discovers network equipment connected to its serial ports, capturing hostname, model, OS version, management IP, and live health telemetry. This is the AOOB value: the console server knows *what* it's managing, not just which port it's on.

In [74]:
get_managed_devices = managed_devices()

if get_managed_devices is not None:
    # pprint(get_managed_devices.json())
    print(f"\nTotal Managed Devices: {get_managed_devices.json()["total_devices"]}...\n")    
    rows = [
        [
            m.get("port_id", "?"),
            m.get("type", "n/a"),
            m.get("model", "n/a"),
            m.get("hostname", "n/a"),
            m.get("os_version"),
            m.get("management_ipv4", "n/a"),
            m.get("temp", "n/a"),
            m.get("cpu_usage", "?"),
            m.get("mamory_usage", "?"),
        ]
        for m in get_managed_devices.json()['devices']
    ]
    headers = ["Port", "Type", "Model", "Hostname", "OS Version", "Mgt IP", 
               "Temp", "CPU Usage", "Memory Usage"]
    print(tabulate(rows, headers=headers, tablefmt="github"))

https://10.40.21.41/api/v2/managed_devices successful (200)

Total Managed Devices: 1...

|   Port | Type      | Model      | Hostname      | OS Version   | Mgt IP      | Temp   |   CPU Usage | Memory Usage   |
|--------|-----------|------------|---------------|--------------|-------------|--------|-------------|----------------|
|      2 | cisco ios | C892FSP-K9 | c892fsp-k9-02 | 15.7(3)M     | 10.40.21.46 | n/a    |           2 | ?              |


### Single port detail, hardware signals and byte counters

The per-port status endpoint returns hardware signal states (CTS, DSR, DTR, RTS) and byte counters alongside the status summary. Change `DEMO_PORT` to inspect any port by number or name.

In [75]:
DEMO_PORT = 2  # change to any port number (1-32) or port name

port_detail = port_status(DEMO_PORT)

if port_detail is not None:
    pprint(port_detail.json())

https://10.40.21.41/api/v2/ports/2/status successful (200)
{'bytes_input': 13362774,
 'bytes_output': 125369,
 'cts': True,
 'dsr': True,
 'dtr': True,
 'errors': 0,
 'id': 2,
 'md_name': '',
 'md_type': '',
 'name': 'C892FSP-K9',
 'rts': True,
 'status': 'Idle',
 'type': 'rj45'}


### Log out and terminate API session

In [76]:
logout = user_logout()

if logout is not None:
    pprint(logout.json())

https://10.40.21.41/api/v2/user/login successful (200)
{'code': 'SUCCESS', 'message': ['API session terminated'], 'status': 200}


### Others...

In [ ]:
get_config_commands = config_commands()

if get_config_commands is not None:
    pprint(get_config_commands.json())

In [ ]:
post_system_reboot = system_reboot()

if post_system_reboot is not None:
    pprint(post_system_reboot.json())

In [23]:
get_cellular = cellular_status()

if get_cellular is not None:
    pprint(get_cellular.json())

https://10.40.21.41/api/v2/cellular/status successful (200)
{'apn': 'null',
 'band': '',
 'carrier': 'null',
 'connection': 'null',
 'country_operator': 'null',
 'current_band': 'null',
 'dns_server1': 'null',
 'dns_server2': 'null',
 'firmware_revision': 'null',
 'gateway': 'null',
 'hardware_revision': 'null',
 'iccid': 'null',
 'imei': 'null',
 'ipv4_address': 'null',
 'ipv4_mask': 'null',
 'ipv6_global': 'null',
 'ipv6_state': True,
 'ipv6_static': 'null',
 'link_state': 'down',
 'model': 'null',
 'modem': False,
 'network_registration': 'null',
 'os_version': 'null',
 'packet_data_state': 'down',
 'pdp_context_id': 1,
 'preferred_network': 'auto',
 'roaming_state': False,
 'roaming_status': False,
 'rx_bytes': 0,
 'rx_errors': 0,
 'rx_multicast': 0,
 'rx_packets': 0,
 'serial_number': 'null',
 'signal_strength': 0,
 'sim_card': 'not_inserted',
 'state': 'DHCP',
 'tx_bytes': 0,
 'tx_errors': 0,
 'tx_packets': 0,
 'uptime': 0}


In [27]:
get_port_status = port_status(port_id=2)

if get_port_status is not None:
    pprint(get_port_status.json())

https://10.40.21.41/api/v2/ports/2/status successful (200)
{'bytes_input': 12579737,
 'bytes_output': 118060,
 'cts': True,
 'dsr': True,
 'dtr': True,
 'errors': 0,
 'id': 2,
 'md_name': '',
 'md_type': '',
 'name': 'C892FSP-K9',
 'rts': True,
 'status': 'Direct to SSH',
 'type': 'rj45'}


In [ ]:
firmware_bootbank_set(payload={"bank": 1})

In [ ]:
user_login(username=USERNAME, password=PASSWORD)

In [ ]:
firmware_bootbank_set(bank=1)

# Working around PUT issue #

## This times out ##

In [ ]:
token = session.headers["X-Auth-Token"]
url = "https://10.40.21.41/api/v2/firmware/bootbank"
body = '{"bank": 1}'

headers = {
    # "Host": "10.40.21.41",
    # "User-Agent": "curl/8.7.1",
    "Content-Type": "application/json",
    # "Accept": "application/json",
    "X-Auth-Token": token,
    # "Accept-Encoding": "identity",  # avoid gzip/deflate differences
    # "Connection": "close",
    # "Content-Length": str(len(body)),
}

response = requests.put(
    url,
    headers=headers,
    data=body,
    timeout=10,
    verify=False,
)

print("Status:", response.status_code)
print("Body:", response.text)
print("Sent headers:\n", response.request.headers)
print("Sent body:\n", response.request.body)

In [ ]:
token = session.headers["X-Auth-Token"]

url = "https://10.40.21.41/api/v2/firmware/bootbank"

payload = { "bank": 1 }
headers = {
    "Content-Type": "application/json",
    "Accept": "application/json",
    "X-auth-token": token
}

response = requests.put(url, json=payload, headers=headers, verify=False,)

print(response.json())

## This works ##

In [ ]:
import subprocess
import json

def firmware_bootbank_set_via_curl(bank: int, token: str):
    cmd = [
        "curl",
        "-k",
        "-X", "PUT",
        "-H", "Content-Type: application/json",
        "-H", "Accept: application/json",
        f"-H", f"X-Auth-Token:{token}",
        "-d", json.dumps({"bank": bank}),
        "https://10.40.21.41/api/v2/firmware/bootbank",
    ]
    result = subprocess.run(cmd, capture_output=True, text=True, check=False)
    return result.stdout, result.stderr, result.returncode

In [ ]:
change_bootbank = firmware_bootbank_set_via_curl(2, session.headers['X-Auth-Token'])
for _ in change_bootbank:
    print(_)